# TrainWatch: Full Demo

This notebook demonstrates all core features in Phase-01.
PyPI distribution name: `trainwatch-notify`.
Import package name: `trainwatch`.

In [ ]:
# If TrainWatch is not installed yet:
# !pip install trainwatch-notify

## Notification setup (recommended)

Most users should use TrainWatch Cloud email (no SMTP).
Advanced SMTP/Telegram config is optional (see below).

In [ ]:
from trainwatch import add_email

# Most users (hosted backend):
# add_email("you@example.com")

## Advanced: SMTP or Telegram (optional)
Use this only if you want to manage your own SMTP/Telegram credentials or self-host the Worker.

In [ ]:
# Optional: local config dict for SMTP/Telegram
cfg = None

# cfg = {
#     "notifications": {"email": True, "telegram": False},
#     "email": {
#         "host": "smtp.example.com",
#         "port": 587,
#         "username": "you@example.com",
#         "password": "YOUR_APP_PASSWORD",
#         "sender": "you@example.com",
#         "recipient": "you@example.com",
#         "use_tls": True,
#         "subject": "TrainWatch Notification",
#     },
# }

# If you define cfg above, pass config=cfg to end()/fail()/notify().

## Basic training run (manual try/except)

In [ ]:
from trainwatch import monitor

def train_model(epochs=3):
    for epoch in range(1, epochs + 1):
        loss = 0.6 / epoch
        val_acc = 0.75 + (epoch * 0.03)
        monitor.log(epoch=epoch, loss=loss, val_acc=val_acc)

monitor.start()
try:
    train_model(epochs=5)
    monitor.end(config=cfg)
except Exception as exc:
    monitor.fail(exc, config=cfg)
    raise

## Context manager (auto end/fail)

In [ ]:
from trainwatch import monitor

def train_quick():
    for epoch in range(1, 4):
        monitor.log(epoch=epoch, loss=0.5 / epoch)

with monitor.watch(config=cfg):
    train_quick()

## Manual milestone notification

In [ ]:
from trainwatch import monitor

monitor.start()
monitor.notify("Model loaded on GPU", config=cfg)
monitor.end(config=cfg)

## Heartbeat and step-based pings

In [ ]:
import time
from trainwatch import monitor

monitor.start()
monitor.heartbeat(interval_seconds=60, message="Training still running", config=cfg)

for epoch in range(1, 4):
    time.sleep(1)
    monitor.step(notify_every=2, message=f"Epoch {epoch} completed", config=cfg)

monitor.end(config=cfg)

## Configure defaults once

In [ ]:
from trainwatch import monitor

monitor.configure(
    heartbeat_interval=120,
    step_notify_every=3,
    heartbeat_message="Still training",
)

monitor.start()
for step in range(1, 7):
    monitor.step()
monitor.end(config=cfg)

## Run log file (optional)

Enable logging to write a JSON summary to disk.

In [ ]:
from trainwatch import monitor

cfg_logging = {
    "logging": {"enabled": True, "path": "trainwatch_run.json"}
}

monitor.start()
monitor.end(config=cfg_logging)

## Failure example

Uncomment the error to trigger a failure notification.

In [ ]:
from trainwatch import monitor

monitor.start()
try:
    # raise RuntimeError("Intentional failure example")
    monitor.end(config=cfg)
except Exception as exc:
    monitor.fail(exc, config=cfg)
    raise

## Cleanup (cloud email)

If you used TrainWatch Cloud, you can remove your email registration.

In [ ]:
from trainwatch import delete_email

# delete_email()

## CLI commands (optional)

You can also use the CLI in a terminal:

```
trainwatch add-email you@example.com
trainwatch delete-email
trainwatch help
```